# 07. 신호 조합 (AND) — 실행 점검

대상: 팀원 4명 (조합 결과를 같이 보기 전에, 결과 파일이 규칙대로 만들어졌는지 확인하는 용도)

규칙은 실행 전에 `실험기록.csv` 17·18번째 줄에 적었다 (기준 5.6·5.7, 11/25 팀장). 결과를 보고 규칙을 바꾸지 않는다.
이 노트북은 **만든 과정**만 확인한다. 조합 점수·순위·합격 여부(합격 시험 1~6)는 여기서 보지 않는다 — 다음 단계에서 따로 본다.

1. 조합 부품 테스트 (AND 날짜 창·미래 엿보기·153개 목록·3신호 고르기 규칙)
2. 점검: `rsi+rsi` 조합 = 단독 `rsi` (A 그리고 A = A 이므로 끝자리까지 같아야 한다)
3. 2신호 153개 실행 점검 (묶음·지문·합친 파일)
4. 실행 중 코드를 덧붙인 일 — 다시 계산해서 같은지
5. 3신호 상위 6개 고르기 (토막 3개 순위, 확인 장치)

이 노트북은 데이터 파일을 저장하지 않는다 (§9). 결과 파일은 명령(`python -m indexbeater combo ...`)만 만든다.

In [1]:
# 이 칸을 맨 먼저 실행한다.
# 노트북이 어느 폴더에서 열려도, indexbeater 폴더가 있는 프로젝트 폴더를 찾아서 길 목록(sys.path)에 넣는다.
import os
import sys

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "indexbeater")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)          # 못 찾으면 한 칸 위 폴더로 올라가서 다시 찾는다
sys.path.insert(0, root)
os.chdir(root)
print("프로젝트 폴더:", root)

프로젝트 폴더: C:\Users\gram\Desktop\project\buysignal


## 1. 조합 부품 테스트

`tests/check_period.py` 중 조합 테스트만 돌린다 (기본 pytest 130개 모음에는 넣지 않는다 — 노트북 04 기준값 때문).
- AND 날짜 창 경계(T−2 는 켜짐, T−3 은 안 켜짐), 자기 자신과 조합 = 단독, 3신호, 미래 엿보기(뒤 날짜를 바꿔도 앞 날짜 조합 신호는 그대로)
- 153개 목록·5.5 고른 세트 읽기, 다른 지문으로 만든 묶음은 섞지 않음
- 3신호: 토막이 cut 을 넘지 않음, 순위·동점 규칙(Edge 없음 = 꼴찌, 9개 중앙값 → 평균 → 부록 A 번호), 20개·816개 목록

맨 아래 줄이 `passed` 이고 `failed` 가 없으면 통과.

In [2]:
import subprocess

r = subprocess.run([sys.executable, "-m", "pytest", "tests/check_period.py", "-k", "combo or triple", "--color=no", "-q"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:] or r.stderr[-1500:])

.........                                                                [100%]



## 2. 점검: `rsi+rsi` = 단독 `rsi`

조합 코드가 단독 점수표와 **같은 계산**을 하는지 본다. 비교 대상은 단독을 돌린 원본 묶음 `results/scorecard_parts/rsi/`
(합친 `results/scorecard.csv` 는 숫자 끝자리가 잘려 있어서 쓰지 않는다 — 차이 약 1e-16, 결과에는 영향 없음).
이름 열(신호·세트)만 빼고 점수표·OOS 매매 하나하나·포트폴리오 일별이 끝자리까지 같아야 한다. 2~3분 걸린다.

In [3]:
r = subprocess.run([sys.executable, "-m", "indexbeater", "combo", "--self-check", "rsi"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print("\n".join(l for l in r.stdout.splitlines() if "매매 계산" not in l)[-2000:] or r.stderr[-2000:])
print("끝 코드:", r.returncode, "(0 이면 통과)")

점검: rsi+rsi 조합 vs C:\Users\gram\Desktop\project\buysignal\results\scorecard_parts\rsi
  ✓ 엔진·점수표·데이터 지문 = 단독 점수표 때와 같음
  대조군 바탕 H=5: 체결 1,503,504건 (28초)
  대조군 바탕 H=10: 체결 1,499,231건 (52초)
  대조군 바탕 H=20: 체결 1,490,408건 (76초)
  ✓ 점수표 같음: 12줄 × 52열 (안 본 열: 신호, 2020용세트)
  ✓ OOS 매매 같음: 9,123줄 × 16열 (안 본 열: 신호, 세트)
  ✓ 포트폴리오 일별 같음: 7,362줄 × 5열 (안 본 열: 신호)
끝 코드: 0 (0 이면 통과)


## 3. 2신호 153개 실행 점검

만든 명령: `python -m indexbeater combo --pairs` (팀장, 11/25, 51분) → `python -m indexbeater combo --merge-pairs`.
- 153개 묶음이 다 있고, 지문(엔진·점수표·데이터)이 단독 점수표 때와 같은가
- 합친 파일이 시장마다 918칸(153 × H 3 × 비용 2, 기준 5.6)인가, 빈칸이 없는가
- 포트폴리오 열은 0.5% 줄에만 (0.25% 는 상위 후보만, 5.6), 지수 초과수익 열이 붙었는가
- 거래 300건 미만 칸 수·표시 개수 (점수 순위는 여기서 보지 않는다)

In [4]:
# 153개 묶음 점검 + 합친 파일 점검 (숫자 순위는 여기서 보지 않는다 — 합격 시험은 4단계)
import json
from pathlib import Path

import pandas as pd

from indexbeater import combo as CB
from indexbeater import config as C

now = CB.engine_fingerprints()
pairs = CB.pair_list()
d = CB.PARTS / "pairs"
metas = [json.loads((d / CB.combo_name(p) / "meta.json").read_text(encoding="utf-8")) for p in pairs]
print("묶음 수:", len(metas), "/ 153")
print("지문이 지금과 같은 묶음:", sum(m["지문"] == now for m in metas))
print("combo.py 판:", sorted({m["combo.py"] for m in metas}))
print("환경:", sorted({json.dumps(m["환경"], sort_keys=True) for m in metas}))
print("git:", sorted({m["git"] for m in metas}))
print(f"걸린 시간 합: {sum(m['걸린초'] for m in metas) / 60:.1f}분 (준비 시간 빼고)")
print("행 수 (점수표 줄):", sorted({m["행수"]["scorecard"] for m in metas}))

RT = dict(encoding="utf-8-sig", float_precision="round_trip")
s = pd.read_csv(CB.PAIRS_OUT, **RT)
print("\n합친 파일:", CB.PAIRS_OUT.name, s.shape)
cells = s.groupby("시장").size()
print("시장별 칸 (조합 × H × 비용):", cells.to_dict(), "→ 기준 5.6 '918칸'과", "같음" if (cells == 918).all() else "다름")
print("조합 이름 153개 다 있음:", sorted(s["신호"].unique()) == sorted(CB.combo_name(p) for p in pairs))
print("점수 빈칸:", int(s["점수"].isna().sum()))
p5, p25 = s[s["비용"] == "0.5%"], s[s["비용"] == "0.25%"]
print("포트폴리오 열: 0.5% 줄 빈칸", int(p5["포트_MDD"].isna().sum()), "/ 0.25% 줄 채움", int(p25["포트_MDD"].notna().sum()))
print("초과수익 열: 0.5% 줄 빈칸", int(p5["포트_초과수익_중앙값"].isna().sum()))
print("거래 300건 미만 칸 (0.5%):", int((p5["거래수"] < C.MIN_TRADES).sum()), "/", len(p5))
flags = p5["표시"].fillna("").str.split(",").explode()
print("표시 개수 (0.5% 줄):", flags[flags != ""].value_counts().to_dict())

묶음 수: 153 / 153
지문이 지금과 같은 묶음: 153
combo.py 판: ['5417d59faf4f', '60857791ed9e', '6fd8c75674b4', '980e6ff5ed52', 'e23340b6ac23']
환경: ['{"numpy": "2.5.2", "pandas": "3.0.5", "pyarrow": "26.0.1", "python": "3.12.10", "scipy": "1.18.1"}']
git: ['f7d6814 (고친 파일 있음)']
걸린 시간 합: 50.7분 (준비 시간 빼고)
행 수 (점수표 줄): [12]

합친 파일: combo_pairs.csv (1836, 76)
시장별 칸 (조합 × H × 비용): {'KOSDAQ': 918, 'KOSPI': 918} → 기준 5.6 '918칸'과 같음
조합 이름 153개 다 있음: True
점수 빈칸: 0
포트폴리오 열: 0.5% 줄 빈칸 0 / 0.25% 줄 채움 0
초과수익 열: 0.5% 줄 빈칸 0
거래 300건 미만 칸 (0.5%): 228 / 918
표시 개수 (0.5% 줄): {'기본값 사용': 615, '경고(승률)': 273, '표본 부족': 228, '경고': 120}


## 4. 실행 중에 combo.py 를 고친 일 — 다시 계산해서 확인

153개가 도는 동안 같은 파일(combo.py)에 3신호 코드를 덧붙였다. 돌던 프로그램은 처음 읽은 코드로 계속 돌았지만,
묶음마다 적는 'combo.py 판'은 **저장하는 순간의 파일**을 읽어서 적었다 → 판이 5가지로 적혔다 (위 3절).
이제는 시작할 때 한 번만 적도록 고쳤다 (11/25).

확인: 판마다 묶음 하나씩 + 마지막 묶음을 **지금 코드로 다시 계산**해서 저장된 묶음과 끝자리까지 비교한다.
모두 같으면 덧붙인 코드는 2신호 계산을 바꾸지 않았고, 153개는 한 가지 계산으로 만들어졌다. 4~5분 걸린다.

In [5]:
# 실행 중에 combo.py 에 3신호 코드를 덧붙였다 → 묶음마다 적힌 'combo.py 판'이 실제로 돈 코드가 아니라
# 저장 순간의 파일이다 (11/25 고침: 이제는 시작할 때 한 번 적는다).
# 그래서 판마다 묶음 하나씩을 지금 코드로 다시 계산해서, 저장된 묶음과 끝자리까지 같은지 본다.
# 같으면: 덧붙인 코드는 2신호 계산을 바꾸지 않았고, 153개는 모두 한 가지 계산으로 만들어졌다.
import io
import json

import pandas as pd

from indexbeater import combo as CB

d = CB.PARTS / "pairs"
metas = {CB.combo_name(p): json.loads((d / CB.combo_name(p) / "meta.json").read_text(encoding="utf-8"))
         for p in CB.pair_list()}
groups = {}
for name, m in metas.items():
    groups.setdefault(m["combo.py"], []).append(name)
print("적힌 판:", {k: len(v) for k, v in groups.items()})
check = [v[0] for v in groups.values()] + [list(metas)[-1]]
check = list(dict.fromkeys(check))
print("다시 계산해 볼 조합:", check)

now, ctx, picks, flags, cache = CB.prepare()


def as_saved(df):
    return pd.read_csv(io.StringIO(df.to_csv(index=False)), float_precision="round_trip")


for name in check:
    got = CB.combo_tables(tuple(name.split(CB.SEP)), ctx, picks, flags, cache, log=lambda *a: None)
    src = d / name
    pd.testing.assert_frame_equal(as_saved(got["scorecard"]),
                                  pd.read_csv(src / "scorecard.csv", encoding="utf-8-sig", float_precision="round_trip"),
                                  check_dtype=False, check_exact=True)
    pd.testing.assert_frame_equal(got["oos_trades"].reset_index(drop=True),
                                  pd.read_parquet(src / "oos_trades.parquet"), check_exact=True)
    pd.testing.assert_frame_equal(got["portfolio_daily"].reset_index(drop=True),
                                  pd.read_parquet(src / "portfolio_daily.parquet"), check_exact=True)
    print(f"  ✓ {name}: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음")

적힌 판: {'5417d59faf4f': 113, 'e23340b6ac23': 5, '6fd8c75674b4': 2, '980e6ff5ed52': 1, '60857791ed9e': 32}
다시 계산해 볼 조합: ['ma_cross+alignment', 'disparity+vol_breakout', 'cci+bollinger', 'cci+donchian', 'cci+vol_breakout', 'obv+candle']


  ✓ 엔진·점수표·데이터 지문 = 단독 점수표 때와 같음


  대조군 바탕 H=5: 체결 1,503,504건 (28초)


  대조군 바탕 H=10: 체결 1,499,231건 (52초)


  대조군 바탕 H=20: 체결 1,490,408건 (75초)


  ✓ ma_cross+alignment: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음


  ✓ disparity+vol_breakout: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음


  ✓ cci+bollinger: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음


  ✓ cci+donchian: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음


  ✓ cci+vol_breakout: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음


  ✓ obv+candle: 점수표·OOS 매매·포트폴리오 일별 모두 저장된 묶음과 같음


## 5. 3신호 상위 6개 고르기 (기준 5.7-1~3, 실험기록 18번째 줄)

만든 명령: `python -m indexbeater combo --triple-pick` (팀장, 11/25, 2분 25초).
- 토막 = 2015년용 최초 학습 구간(신호 시작 ~ cut 2014-12-01)을 거래일 수로 3등분 → 2015년 가격은 쓰지 않는다
- 매매·Edge = 5.5 학습 규칙 그대로 → **확인 장치**: 세 토막 합친 Edge·거래 수 = 단독 점수표 2015 학습Edge·학습거래수 (108칸, 끝자리까지).
  하나라도 다르면 명령이 파일을 저장하지 않는다. 여기서 파일을 다시 읽어 한 번 더 확인한다.
- 시장마다 순위 9개(토막 3 × H 3)의 중앙값 → 상위 6개 → C(6,3) = 20개

읽는 법: '순위중앙값' 이 작을수록 세 토막·세 H 에서 고르게 앞섰다는 뜻이다. 한 토막에서만 1위인 신호는 중앙값이 나빠서 뽑히지 않는다
(한 시기에만 잘 맞은 신호를 거르려는 규칙, 과적합 방지). 이것은 '고르기'일 뿐이고, 좋은지는 2015~2019 OOS 점수로 따로 본다.

In [6]:
# 3신호 고르기 결과 파일 읽기 (만든 명령: python -m indexbeater combo --triple-pick)
# 확인 장치(세 토막 합친 Edge = 단독 2015 학습Edge, 108칸 끝자리까지)는 명령 안에서 통과해야만 파일이 저장된다.
# 여기서는 파일을 다시 읽어 같은 확인을 한 번 더 하고, 시장별 상위 6개와 20개 목록을 보여 준다.
import pandas as pd

from indexbeater import combo as CB
from indexbeater import config as C

seg = pd.read_csv(CB.TRIPLE_SEG_OUT, encoding="utf-8-sig", float_precision="round_trip")
top = pd.read_csv(CB.TRIPLE_RANK_OUT, encoding="utf-8-sig", float_precision="round_trip")

print("토막 날짜:")
print(seg[seg["토막"] != "전체"].groupby("토막")[["처음", "끝"]].first().to_string())
bad = CB.check_segments(seg)
print("다시 확인 (전체 = 단독 2015 학습Edge·거래 수, 토막 합 = 전체):", "✓ 모두 같음" if not bad else bad[:5])
print("거래 0건 칸 (토막 × H):", int(((seg["토막"] != "전체") & (seg["거래수"] == 0)).sum()))

for mk in C.UNIVERSE_MARKETS:
    t = top[top["시장"] == mk]
    print(f"\n[{mk}] 18개 순위 (순위 9개 = 토막 3 × H 3, 작을수록 좋음)")
    print(t[["등수", "신호", "순위중앙값", "순위평균", "상위"]].to_string(index=False))
    tri = CB.triple_list(top, mk)
    print(f"→ 3신호 {len(tri)}개:", ", ".join(CB.combo_name(x) for x in tri))

토막 날짜:
            처음           끝
토막                        
1   2012-07-02  2013-04-17
2   2013-04-18  2014-02-10
3   2014-02-11  2014-12-01


다시 확인 (전체 = 단독 2015 학습Edge·거래 수, 토막 합 = 전체):

 ✓ 모두 같음
거래 0건 칸 (토막 × H): 0

[KOSPI] 18개 순위 (순위 9개 = 토막 3 × H 3, 작을수록 좋음)
 등수              신호  순위중앙값      순위평균    상위
  1 volume_breakout    1.0  5.000000  True
  2       disparity    4.0  3.888889  True
  3             rsi    5.0  5.444444  True
  4        donchian    5.0  7.222222  True
  5         squeeze    6.0  6.888889  True
  6          adx_di    7.0  7.111111  True
  7        ichimoku    7.0  8.444444 False
  8        ma_cross    7.0 10.555556 False
  9       alignment    8.0  9.000000 False
 10       bollinger    9.0  8.888889 False
 11             cci    9.0 11.555556 False
 12            macd   11.0 10.555556 False
 13             sar   12.0 11.333333 False
 14    vol_breakout   12.0 12.000000 False
 15          candle   13.0 11.888889 False
 16      stochastic   15.0 12.666667 False
 17             obv   15.0 14.222222 False
 18      williams_r   15.0 14.333333 False
→ 3신호 20개: adx_di+rsi+disparity, adx_di+rsi+squeeze, adx_di+rsi+donchian, adx_di+rsi+volume_breakout, adx_di

## 6. 후보 0건 칸 = 포트폴리오 빈칸 (실험기록 21·24번째 줄)

3신호를 돌리다가 **매수 후보가 하루도 없는 조합**에서 프로그램이 멈췄다 (AND 로 묶으면 세 신호가 3일 안에 함께 켜지는 날이 아예 없을 수 있다).
- 고친 것: `combo.py` `has_portfolio_candidate` — 후보가 0건이면 포트폴리오 열을 **빈칸**으로 두고, meta.json 에 `'포트폴리오없음(후보0건)'` 으로 어느 칸인지 적는다 (사용자 결정 "1번").
  0.0(수익 0%)으로 적으면 "돈을 안 잃었다"로 읽혀서 순위에 끼어들 수 있기 때문.
- 고치기 전 판으로 만든 묶음 3개에는 같은 칸이 0.0 으로 적혀 있었다 (8칸) → 사용자 결정 "1번" = 그 3묶음만 다시 돌리고 `combo_pairs.csv` 를 다시 합쳤다.
  다시 돌린 뒤 바뀐 것은 그 8칸의 포트폴리오 열뿐이고, 매매 기록·나머지 칸은 끝자리까지 같았다.
- 그래서 **3절 출력은 다시 돌리기 전 것**이다 (3절의 '0.5% 줄 빈칸 0', 'combo.py 판 5가지'). 지금 상태는 아래 칸에서 본다.

In [2]:
# 지금 결과 파일에서: 포트폴리오 빈칸 = 거래 0건 칸 = meta '포트폴리오없음(후보0건)' 인지, 옛 0.0 8칸이 빈칸이 됐는지 본다.
import json

import pandas as pd

from indexbeater import combo as CB
from indexbeater import stage4 as S4

RT = dict(encoding="utf-8-sig", float_precision="round_trip")
for kind, out in (("pairs", CB.PAIRS_OUT), ("triples", CB.TRIPLES_OUT)):
    s = pd.read_csv(out, **RT)
    p5 = s[s["비용"] == "0.5%"]
    blank = p5["포트_수익_2015"].isna()
    zero = p5["거래수"] == 0
    metas = [json.loads(f.read_text(encoding="utf-8")) for f in sorted((CB.PARTS / kind).glob("*/meta.json"))]
    meta_n = sum(len(m.get("포트폴리오없음(후보0건)", [])) for m in metas)
    print(f"[{out.name}] 0.5% 칸 {len(p5):,} / 포트폴리오 빈칸 {int(blank.sum())} / 거래 0건 {int(zero.sum())} / "
          f"meta 적힌 칸 {meta_n} / 빈칸 = 거래 0건: {bool((blank == zero).all())} / 거래 0건인데 0.0: {int((zero & ~blank).sum())}")
    print("   시장별 빈칸:", p5[blank].groupby("시장").size().to_dict())

both = pd.concat([pd.read_csv(CB.PAIRS_OUT, **RT).assign(종류="2신호"),
                  pd.read_csv(CB.TRIPLES_OUT, **RT).assign(종류="3신호")])
both = both[both["비용"] == "0.5%"]
old = both[[(k, n, m, int(h)) in S4.OLD_ZERO_CELLS for k, n, m, h in zip(both["종류"], both["신호"], both["시장"], both["H"])]]
print(f"\n옛 0.0 칸 {len(S4.OLD_ZERO_CELLS)}개 → 지금 파일:")
print(old[["종류", "신호", "시장", "H", "거래수", "포트_수익_2015", "포트_MDD"]].to_string(index=False))

[combo_pairs.csv] 0.5% 칸 918 / 포트폴리오 빈칸 3 / 거래 0건 3 / meta 적힌 칸 3 / 빈칸 = 거래 0건: True / 거래 0건인데 0.0: 0
   시장별 빈칸: {'KOSPI': 3}


[combo_triples.csv] 0.5% 칸 4,896 / 포트폴리오 빈칸 203 / 거래 0건 203 / meta 적힌 칸 203 / 빈칸 = 거래 0건: True / 거래 0건인데 0.0: 0
   시장별 빈칸: {'KOSDAQ': 72, 'KOSPI': 131}



옛 0.0 칸 8개 → 지금 파일:
 종류                             신호     시장  H  거래수  포트_수익_2015  포트_MDD
2신호                   ichimoku+rsi  KOSPI  5    0         NaN     NaN
2신호                   ichimoku+rsi  KOSPI 10    0         NaN     NaN
2신호                   ichimoku+rsi  KOSPI 20    0         NaN     NaN
3신호 alignment+stochastic+disparity  KOSPI  5    0         NaN     NaN
3신호 alignment+stochastic+disparity  KOSPI 10    0         NaN     NaN
3신호 alignment+stochastic+disparity  KOSPI 20    0         NaN     NaN
3신호          rsi+disparity+squeeze KOSDAQ  5    0         NaN     NaN
3신호          rsi+disparity+squeeze KOSDAQ 20    0         NaN     NaN


## 7. 3신호 816개 실행 점검 (기준 5.7-6, 실험기록 17·22·26번째 줄)

만든 명령: `python -m indexbeater combo --triples-all` (팀장, 11/25, 9,579초 = 약 2시간 40분, 오류 0) → `python -m indexbeater combo --merge-triples`.
- 816개 = 18개 신호 중 3개를 고르는 모든 경우 (C(18,3)). **비교만 한다** — 순위·개선 판정은 5절에서 고른 20개로만 (사용자 결정 "2번").
- 합친 파일 `results/combo_triples.csv` 의 '고른20' 열 = 그 시장에서 5절 규칙으로 고른 20개에 드는 칸.
- 묶음 816개가 다 있고, 지문(엔진·점수표·데이터)이 단독 점수표 때와 같은지, combo.py 판이 몇 가지인지 본다 (판이 2가지 = 25개는 6절 고치기 전 판, 9절에서 다시 계산해 확인).

In [3]:
import json

import pandas as pd

from indexbeater import combo as CB
from indexbeater import config as C

now = CB.engine_fingerprints()
tri = CB.all_triples()
d = CB.PARTS / "triples"
metas = [json.loads((d / CB.combo_name(t) / "meta.json").read_text(encoding="utf-8")) for t in tri]
print("묶음 수:", len(metas), "/ 816")
print("지문이 지금과 같은 묶음:", sum(m["지문"] == now for m in metas))
print("combo.py 판:", pd.Series([m["combo.py"] for m in metas]).value_counts().to_dict())
print("환경:", sorted({json.dumps(m["환경"], sort_keys=True) for m in metas}))
print(f"걸린 시간 합: {sum(m['걸린초'] for m in metas) / 3600:.2f}시간 (준비 시간 빼고)")

RT = dict(encoding="utf-8-sig", float_precision="round_trip")
s = pd.read_csv(CB.TRIPLES_OUT, **RT)
print("\n합친 파일:", CB.TRIPLES_OUT.name, s.shape)
print("시장별 줄 (816 × H3 × 비용2 = 4,896):", s.groupby("시장").size().to_dict())
print("조합 이름 816개 다 있음:", sorted(s["신호"].unique()) == sorted(CB.combo_name(t) for t in tri))
p5 = s[s["비용"] == "0.5%"]
print("'고른20' 칸 (0.5% 줄, 시장마다 20 × H3 = 60):", p5[p5["고른20"]].groupby("시장").size().to_dict())
print("점수 빈칸:", int(s["점수"].isna().sum()))
print("거래 300건 미만 칸 (0.5%):", p5[p5["거래수"] < C.MIN_TRADES].groupby("시장").size().to_dict(), "/ 시장마다", len(p5) // 2)
flags = p5["표시"].fillna("").str.split(",").explode()
print("표시 개수 (0.5% 줄):", flags[flags != ""].value_counts().to_dict())

묶음 수: 816 / 816
지문이 지금과 같은 묶음: 816
combo.py 판: {'e450fce85649': 791, '60857791ed9e': 25}
환경: ['{"numpy": "2.5.2", "pandas": "3.0.5", "pyarrow": "26.0.1", "python": "3.12.10", "scipy": "1.18.1"}']
걸린 시간 합: 2.76시간 (준비 시간 빼고)

합친 파일: combo_triples.csv (9792, 78)
시장별 줄 (816 × H3 × 비용2 = 4,896): {'KOSDAQ': 4896, 'KOSPI': 4896}
조합 이름 816개 다 있음: True
'고른20' 칸 (0.5% 줄, 시장마다 20 × H3 = 60): {'KOSDAQ': 60, 'KOSPI': 60}
점수 빈칸: 0
거래 300건 미만 칸 (0.5%): {'KOSDAQ': 1488, 'KOSPI': 1633} / 시장마다 2448
표시 개수 (0.5% 줄): {'기본값 사용': 4012, '표본 부족': 3121, '경고(승률)': 1928, '경고': 1048, '보류': 16}


## 8. 4단계 합격 시험 결과 (규칙 = 실험기록 22번째 줄, 결과 = 23·25·26·27번째 줄)

규칙은 **조합 순위·점수를 보기 전에** 22번째 줄에 적었다. 모두 2015~2019 OOS(설계 구간의 '처음 보는' 해), 비용 0.5%, 시장별. 2020년 이후 자료는 열지 않았다 (시험 8 은 6단계 뒤 한 번만).

| 시험 | 명령 | 결과 파일 | 판정 |
|---|---|---|---|
| 3 개선 (조합 > 단독 1위 그리고 > 구성 신호) | `stage4 --rank` | stage4_improved · stage4_triple_vs_parts | 코스피 0개, 코스닥 2개 (모두 2신호) |
| 상위 3개 (후보 = 단독 + 개선 조합) | `stage4 --rank` | stage4_pool · stage4_top3 | 아래 표 |
| 7 지수 이기기 (5년 합친 수익 > 비교 지수) | `stage4 --rank` | stage4_top3 | 코스피 후보 A 못 이김 / 코스닥 후보 A 이김 |
| 6 비용 0.25% 로 바꿔 보기 | `stage4 --cost` | stage4_cost | 비용 때문에 판정이 뒤집히는 칸 없음 |
| 4 랜덤 매수 1,000번 (후보 A) | `stage4 --random` | stage4_random(_trials) | 두 시장 모두 5해 중 5해 이김 = 통과 |
| 1 시도 수 | `stage4 --tries` | stage4_tries | 시장마다 6,354 (고른 20개 120칸은 816개 안에 있어 한 번만 셈, 실험기록 28 / 팀원 몫은 6단계) |
| 2 SPA·RC [중간] | `stage4 --spa` | stage4_spa(_cells) | p ≈ 0.99 → '운보다 낫다'는 근거 없음 |
| 5 AND 날짜 창 W=2·4 (보고만) | `stage4 --window` | stage4_window | 코스닥 1위 안정, 2위는 W 에 흔들림 |

아래 칸은 결과 파일을 읽어 보여 주기만 한다 (다시 계산하지 않음). 다시 계산 확인은 각 명령 안에 있다: 예) 랜덤 시험 전 후보 A 포트폴리오를 다시 만들어 저장값과 같은지, W=3 다시 계산 = 저장값.

In [2]:
import pandas as pd

from indexbeater import stage4 as S4

pd.set_option("display.width", 200)
R = lambda p: pd.read_csv(p, encoding="utf-8-sig", float_precision="round_trip")
pct = lambda x: f"{x:+.1%}" if pd.notna(x) else ""

vs = R(S4.OUT_TRIPLE_VS)
imp = R(S4.OUT_IMPROVED)
print("[시험 3] 개선 조합 수 (2신호 153 + 3신호 20, 거래 300건 이상 칸만):",
      imp[imp["개선"]].groupby("시장").size().reindex(["KOSPI", "KOSDAQ"], fill_value=0).to_dict())
print("        개선된 칸:", imp[imp["개선"]][["시장", "신호", "H", "점수"]].to_dict("records"))
print("        3신호 20개 중 구성 신호보다 나은 칸:", vs.groupby("시장")["구성보다나음"].sum().to_dict(),
      "/ 그중 개선:", vs.groupby("시장")["개선"].sum().to_dict())

top = R(S4.OUT_TOP)
t = top[["시장", "등수", "종류", "신호", "H", "점수", "거래수", "포트_5년수익", "지수_5년수익", "지수이긴해수", "포트_MDD", "일관성부족"]].copy()
for c in ("포트_5년수익", "지수_5년수익", "포트_MDD"):
    t[c] = t[c].map(pct)
print("\n[상위 3개 + 시험 7] (1위 = 후보 A)")
print(t.to_string(index=False))

cost = R(S4.OUT_COST)
c = cost[["시장", "등수", "신호", "H", "0.5%_5년수익", "0.25%_5년수익", "지수_5년수익", "비용가정의존"]].copy()
for k in ("0.5%_5년수익", "0.25%_5년수익", "지수_5년수익"):
    c[k] = c[k].map(pct)
print("\n[시험 6] 비용 0.5% → 0.25%")
print(c.to_string(index=False))

rnd = R(S4.OUT_RANDOM)
r = rnd[["시장", "후보A", "구간", "A수익", "랜덤중앙값", "백분위", "이긴해수", "통과(4해이상)"]].copy()
for k in ("A수익", "랜덤중앙값"):
    r[k] = r[k].map(pct)
print("\n[시험 4] 랜덤 매수 1,000번")
print(r.to_string(index=False))

tries = R(S4.OUT_TRIES)
print("\n[시험 1] 시도 수")
print(tries.pivot(index="항목", columns="시장", values="수")[["KOSPI", "KOSDAQ"]].reindex(tries["항목"].unique()).to_string())

spa = R(S4.OUT_SPA)
cells = R(S4.OUT_SPA_CELLS)
print("\n[시험 2] SPA·RC [중간]")
print(spa[["시장", "칸수", "날짜수", "SPA_p", "RC_p", "표준화1위칸"]].to_string(index=False))
print("   칸 종류별 수:", cells.groupby(["시장", "종류"]).size().to_dict())
print("   일평균 초과수익 > 0 인 칸:", cells[cells["일평균초과"] > 0].groupby("시장").size().to_dict(),
      "/ 칸별 p < 0.05:", int((cells["칸별_p"] < 0.05).sum()))
print(cells[cells["상위3등수"].notna()].sort_values(["시장", "상위3등수"])[["시장", "상위3등수", "칸", "일평균초과", "칸별_p", "상위3_Holm_p"]].to_string(index=False))

w = R(S4.OUT_WINDOW)
w = w[["시장", "등수", "신호", "H", "W", "점수", "거래수", "포트_5년수익", "지수_5년수익", "지수이긴해수"]].copy()
for k in ("포트_5년수익", "지수_5년수익"):
    w[k] = w[k].map(pct)
print("\n[시험 5] AND 날짜 창 W (보고만 — 고르기에 안 씀, W=3 은 결과 보기 전에 정한 값)")
print(w.to_string(index=False))

[시험 3] 개선 조합 수 (2신호 153 + 3신호 20, 거래 300건 이상 칸만): {'KOSPI': 0, 'KOSDAQ': 2}
        개선된 칸: [{'시장': 'KOSDAQ', '신호': 'alignment+cci', 'H': 5, '점수': 75.9}, {'시장': 'KOSDAQ', '신호': 'rsi+stochastic', 'H': 5, '점수': 78.9}]
        3신호 20개 중 구성 신호보다 나은 칸: {'KOSDAQ': 2, 'KOSPI': 0} / 그중 개선: {'KOSDAQ': 0, 'KOSPI': 0}

[상위 3개 + 시험 7] (1위 = 후보 A)
    시장  등수  종류             신호  H   점수   거래수 포트_5년수익 지수_5년수익  지수이긴해수 포트_MDD  일관성부족
 KOSPI   1  단독      disparity  5 78.1  5635  -10.9%  +32.3%       2 -38.0%   True
 KOSPI   2  단독            rsi 20 39.7   893  -22.0%  +32.3%       1 -31.0%   True
 KOSPI   3  단독      bollinger 20 37.7 14444  -48.3%  +32.3%       1 -64.7%   True
KOSDAQ   1 2신호 rsi+stochastic  5 78.9   945  +56.7%  +25.8%       3 -13.1%  False
KOSDAQ   2 2신호  alignment+cci  5 75.9   648   +2.8%  +25.8%       3 -21.4%  False
KOSDAQ   3  단독      disparity  5 75.0 12182  -53.7%  +25.8%       1 -66.6%   True

[시험 6] 비용 0.5% → 0.25%
    시장  등수             신호  H 0.5%_5년수익 0.25%_5년수익 지수_5년수익  비용가정의존
